# Clases y objetos · Descriptores y metaclases

Lo que hay debajo de los atributos y de las clases en Python: el protocolo de descriptores (`__get__`, `__set__`, `__set_name__`), con el que están hechos `property`, los métodos y `functools.cached_property`; `__slots__`; los ganchos `__getattr__` y `__getattribute__`; `__init_subclass__` y las metaclases, que intervienen al **crear la clase** y no sus instancias; `__class_getitem__`; y un gestor de contexto que decide si suprime la excepción. El *singleton* con `__new__` y con metaclase y los objetos invocables con `__call__` ya están en *Patrones y SOLID* (ejercicios 4 y 5), y el gestor de contexto que deshace cambios sin suprimir la excepción, en *Python moderno* (ejercicio 11).

## 1. Un descriptor mínimo y `__set_name__`

Escribir un descriptor `Trazado` que guarde el valor en el `__dict__` de la instancia y anote cada lectura y escritura. Usar `__set_name__` para que sepa con qué nombre se declaró sin repetirlo, y comprobar que al leerlo desde la clase (`instancia is None`) devuelve el propio descriptor.

In [1]:
class Trazado:
    def __set_name__(self, propietario, nombre):
        self.nombre = nombre
        self.registro = []

    def __get__(self, instancia, propietario=None):
        if instancia is None:
            return self
        valor = instancia.__dict__[self.nombre]
        self.registro.append(("lee", self.nombre, valor))
        return valor

    def __set__(self, instancia, valor):
        self.registro.append(("escribe", self.nombre, valor))
        instancia.__dict__[self.nombre] = valor


class Guagua:
    linea = Trazado()
    plazas = Trazado()

    def __init__(self, linea, plazas):
        self.linea = linea
        self.plazas = plazas


g = Guagua(12, 55)
g.plazas -= 3
for apunte in Guagua.plazas.registro:
    print(apunte)
print(type(Guagua.__dict__["linea"]).__name__, Guagua.linea.nombre, vars(g))
assert isinstance(Guagua.linea, Trazado) and Guagua.linea.nombre == "linea"
assert Guagua.plazas.registro == [("escribe", "plazas", 55), ("lee", "plazas", 55), ("escribe", "plazas", 52)]

('escribe', 'plazas', 55)
('lee', 'plazas', 55)
('escribe', 'plazas', 52)
Trazado linea {'linea': 12, 'plazas': 52}


## 2. Descriptores de datos y de no datos

Un descriptor con `__set__` (o `__delete__`) es **de datos** y tiene prioridad sobre el `__dict__` de la instancia; uno con solo `__get__` es **de no datos** y la instancia lo puede tapar. Comprobarlo escribiendo a mano en `vars(obj)`, y ver que las funciones son descriptores de no datos: `funcion.__get__(obj)` es exactamente el método ligado.

In [2]:
class DeDatos:
    def __get__(self, instancia, propietario=None):
        return "valor del descriptor de datos"

    def __set__(self, instancia, valor):
        raise AttributeError("solo lectura")


class DeNoDatos:
    def __get__(self, instancia, propietario=None):
        return "valor del descriptor de no datos"


class Prueba:
    a = DeDatos()
    b = DeNoDatos()

    def saludar(self):
        return f"hola desde {type(self).__name__}"


p = Prueba()
vars(p).update(a="valor de la instancia", b="valor de la instancia")
print(f"{'a (de datos)':<16} -> {p.a}")
print(f"{'b (de no datos)':<16} -> {p.b}")
assert p.a == "valor del descriptor de datos" and p.b == "valor de la instancia"

funcion = Prueba.__dict__["saludar"]
ligado = funcion.__get__(p, Prueba)
print(type(funcion).__name__, "->", type(ligado).__name__, "->", ligado())
print("hasattr(funcion, '__set__'):", hasattr(funcion, "__set__"))
assert ligado == p.saludar and ligado.__self__ is p and not hasattr(funcion, "__set__")

a (de datos)     -> valor del descriptor de datos
b (de no datos)  -> valor de la instancia
function -> method -> hola desde Prueba
hasattr(funcion, '__set__'): False


## 3. Atributos validados

Crear una clase base `Validador` (descriptor de datos con un método abstracto `validar`) y dos validadores reutilizables: `Tipo(tipo)` y `Rango(minimo, maximo)`, que también comprueba que sea número. Usarlos en una `Estacion` meteorológica (nombre, temperatura en °C y humedad en %) y comprobar que los valores incorrectos se rechazan tanto en el constructor como en asignaciones posteriores.

In [3]:
from abc import ABC, abstractmethod


class Validador(ABC):
    def __set_name__(self, propietario, nombre):
        self.privado = "_" + nombre
        self.publico = nombre

    def __get__(self, instancia, propietario=None):
        return self if instancia is None else getattr(instancia, self.privado)

    def __set__(self, instancia, valor):
        self.validar(valor)
        setattr(instancia, self.privado, valor)

    @abstractmethod
    def validar(self, valor): ...


class Tipo(Validador):
    def __init__(self, tipo):
        self.tipo = tipo

    def validar(self, valor):
        if not isinstance(valor, self.tipo):
            raise TypeError(f"{self.publico} debe ser {self.tipo.__name__}, no {type(valor).__name__}")


class Rango(Validador):
    def __init__(self, minimo, maximo):
        self.minimo, self.maximo = minimo, maximo

    def validar(self, valor):
        if isinstance(valor, bool) or not isinstance(valor, int | float):
            raise TypeError(f"{self.publico} debe ser un número")
        if not self.minimo <= valor <= self.maximo:
            raise ValueError(f"{self.publico}={valor} fuera de [{self.minimo}, {self.maximo}]")


class Estacion:
    nombre = Tipo(str)
    temperatura = Rango(-30, 60)
    humedad = Rango(0, 100)

    def __init__(self, nombre, temperatura, humedad):
        self.nombre, self.temperatura, self.humedad = nombre, temperatura, humedad

    def __repr__(self):
        return f"Estacion({self.nombre!r}, {self.temperatura} °C, {self.humedad} %)"


izana = Estacion("Izaña", 8.5, 35)
print(izana)
errores = []
for args in [("Teide", 70, 20), ("Teide", 5, -1), (2024, 5, 20), ("Teide", "frío", 20), ("Teide", True, 20)]:
    try:
        Estacion(*args)
    except (TypeError, ValueError) as e:
        errores.append(type(e).__name__)
        print(f"{str(args):<22} {type(e).__name__}: {e}")
try:
    izana.humedad = 120
except ValueError as e:
    print("asignación:", e)
assert errores == ["ValueError", "ValueError", "TypeError", "TypeError", "TypeError"]
assert izana.humedad == 35 and vars(izana) == {"_nombre": "Izaña", "_temperatura": 8.5, "_humedad": 35}

Estacion('Izaña', 8.5 °C, 35 %)
('Teide', 70, 20)      ValueError: temperatura=70 fuera de [-30, 60]
('Teide', 5, -1)       ValueError: humedad=-1 fuera de [0, 100]
(2024, 5, 20)          TypeError: nombre debe ser str, no int
('Teide', 'frío', 20)  TypeError: temperatura debe ser un número
('Teide', True, 20)    TypeError: temperatura debe ser un número
asignación: humedad=120 fuera de [0, 100]


## 4. Propiedad perezosa con caché

Escribir un descriptor de no datos `perezosa` que calcule el valor la primera vez y lo guarde en el `__dict__` de la instancia con el mismo nombre, de modo que las siguientes lecturas ni siquiera pasen por el descriptor. Usarlo para el número de palabras distintas de un texto largo, contar cuántas veces se calcula, invalidar la caché con `del` y comparar con `functools.cached_property`.

In [4]:
from functools import cached_property


class perezosa:
    def __init__(self, funcion):
        self.funcion = funcion
        self.__doc__ = funcion.__doc__

    def __set_name__(self, propietario, nombre):
        self.nombre = nombre

    def __get__(self, instancia, propietario=None):
        if instancia is None:
            return self
        valor = self.funcion(instancia)
        instancia.__dict__[self.nombre] = valor  # tapa al descriptor: no tiene __set__
        return valor


calculos = {"perezosa": 0, "cached_property": 0}


class Texto:
    def __init__(self, contenido):
        self.contenido = contenido

    @perezosa
    def vocabulario(self):
        calculos["perezosa"] += 1
        return len(set(self.contenido.lower().split()))

    @cached_property
    def vocabulario_std(self):
        calculos["cached_property"] += 1
        return len(set(self.contenido.lower().split()))


t = Texto("el mar el viento la arena el mar la calima " * 10_000)
lecturas = [t.vocabulario for _ in range(5)] + [t.vocabulario_std for _ in range(5)]
print("lecturas:", lecturas[0], "| cálculos:", calculos)
print("en vars(t):", sorted(k for k in vars(t) if k != "contenido"))
del t.vocabulario
t.vocabulario
print("tras del, cálculos:", calculos)
assert set(lecturas) == {6} and calculos == {"perezosa": 2, "cached_property": 1}

lecturas: 6 | cálculos: {'perezosa': 1, 'cached_property': 1}
en vars(t): ['vocabulario', 'vocabulario_std']
tras del, cálculos: {'perezosa': 2, 'cached_property': 1}


## 5. `property` reescrita como descriptor

Implementar `Propiedad`, una versión de `property` con `fget`, `fset`, `fdel`, los decoradores `.setter` y `.deleter` (que devuelven una copia nueva, como el original) y `AttributeError` cuando falta una de las tres funciones. Usar la misma clase con `property` y con `Propiedad` y comprobar que se comportan igual.

In [5]:
class Propiedad:
    def __init__(self, fget=None, fset=None, fdel=None, doc=None):
        self.fget, self.fset, self.fdel = fget, fset, fdel
        self.__doc__ = doc or (fget.__doc__ if fget else None)

    def __set_name__(self, propietario, nombre):
        self.nombre = nombre

    def __get__(self, instancia, propietario=None):
        if instancia is None:
            return self
        if self.fget is None:
            raise AttributeError(f"la propiedad {self.nombre!r} no se puede leer")
        return self.fget(instancia)

    def __set__(self, instancia, valor):
        if self.fset is None:
            raise AttributeError(f"la propiedad {self.nombre!r} no tiene setter")
        self.fset(instancia, valor)

    def __delete__(self, instancia):
        if self.fdel is None:
            raise AttributeError(f"la propiedad {self.nombre!r} no tiene deleter")
        self.fdel(instancia)

    def getter(self, fget):
        return type(self)(fget, self.fset, self.fdel, self.__doc__)

    def setter(self, fset):
        return type(self)(self.fget, fset, self.fdel, self.__doc__)

    def deleter(self, fdel):
        return type(self)(self.fget, self.fset, fdel, self.__doc__)


def crear_termometro(decorador):
    class Termometro:
        def __init__(self, celsius):
            self._celsius = celsius

        @decorador
        def celsius(self):
            """Temperatura en grados Celsius."""
            return self._celsius

        @celsius.setter
        def celsius(self, valor):
            if valor < -273.15:
                raise ValueError("por debajo del cero absoluto")
            self._celsius = valor

        @decorador
        def fahrenheit(self):
            return self._celsius * 9 / 5 + 32

    return Termometro


def probar(Termometro):
    t = Termometro(22)
    salida = [t.celsius, t.fahrenheit]
    t.celsius = 30
    salida.append(t.fahrenheit)
    for accion in (lambda: setattr(t, "celsius", -300), lambda: setattr(t, "fahrenheit", 0), lambda: delattr(t, "celsius")):
        try:
            accion()
        except (ValueError, AttributeError) as e:
            salida.append(type(e).__name__)
    salida.append(Termometro.celsius.__doc__)
    return salida


original, propia = probar(crear_termometro(property)), probar(crear_termometro(Propiedad))
print("property :", original)
print("Propiedad:", propia)
assert original == propia

property : [22, 71.6, 86.0, 'ValueError', 'AttributeError', 'AttributeError', 'Temperatura en grados Celsius.']
Propiedad: [22, 71.6, 86.0, 'ValueError', 'AttributeError', 'AttributeError', 'Temperatura en grados Celsius.']


## 6. `__slots__`: memoria y límites

Comparar la memoria de 100 000 puntos con `__dict__` y con `__slots__`, con `sys.getsizeof` (que no cuenta el diccionario) y con `tracemalloc` (que mide todo lo reservado). Comprobar después lo que impiden los `__slots__`: añadir atributos nuevos, las referencias débiles si no se declara `__weakref__`, un valor por defecto de clase con el mismo nombre, y que una subclase sin `__slots__` vuelve a tener `__dict__`.

In [6]:
import sys
import tracemalloc
import weakref


class PuntoDict:
    def __init__(self, x, y):
        self.x, self.y = x, y


class PuntoSlots:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x, self.y = x, y


def memoria(clase, n=100_000):
    tracemalloc.start()
    puntos = [clase(i, -i) for i in range(n)]
    usada, _ = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return usada / n, puntos[0]


print(f"{'clase':<11}{'getsizeof':>10}{'con __dict__':>14}{'tracemalloc/obj':>17}")
medidas = {}
for clase in (PuntoDict, PuntoSlots):
    por_objeto, p = memoria(clase)
    con_dict = sys.getsizeof(p) + (sys.getsizeof(vars(p)) if hasattr(p, "__dict__") else 0)
    medidas[clase.__name__] = por_objeto
    print(f"{clase.__name__:<11}{sys.getsizeof(p):>10}{con_dict:>14}{por_objeto:>15.0f} B")
print(f"__slots__ ahorra un {1 - medidas['PuntoSlots'] / medidas['PuntoDict']:.0%}")
assert medidas["PuntoSlots"] < medidas["PuntoDict"]

clase       getsizeof  con __dict__  tracemalloc/obj
PuntoDict          48           136            160 B


PuntoSlots         48            48            120 B
__slots__ ahorra un 25%


In [7]:
p = PuntoSlots(1, 2)
pruebas = {
    "atributo nuevo": lambda: setattr(p, "z", 3),
    "weakref": lambda: weakref.ref(p),
}
for nombre, prueba in pruebas.items():
    try:
        prueba()
    except (AttributeError, TypeError) as e:
        print(f"{nombre:<15} {type(e).__name__}: {e}")

try:
    class ConDefecto:
        __slots__ = ("color",)
        color = "rojo"
except ValueError as e:
    print(f"{'por defecto':<15} ValueError: {e}")


class PuntoDebil(PuntoSlots):
    __slots__ = ("__weakref__",)


class PuntoHijo(PuntoSlots):
    pass


print("weakref con __weakref__:", weakref.ref(PuntoDebil(0, 0)) is not None)
h = PuntoHijo(1, 2)
h.z = 3
print("subclase sin __slots__ tiene __dict__:", vars(h))
assert not hasattr(p, "__dict__") and vars(h) == {"z": 3}

atributo nuevo  AttributeError: 'PuntoSlots' object has no attribute 'z' and no __dict__ for setting new attributes
weakref         TypeError: cannot create weak reference to 'PuntoSlots' object
por defecto     ValueError: 'color' in __slots__ conflicts with class variable
weakref con __weakref__: True
subclase sin __slots__ tiene __dict__: {'z': 3}


## 7. `__getattr__` frente a `__getattribute__`

`__getattribute__` se llama en **todos** los accesos a atributos; `__getattr__` solo cuando la búsqueda normal falla. Crear una `Configuracion` que devuelva valores por defecto con `__getattr__` y una clase que cuente todos los accesos con `__getattribute__`, delegando en `object.__getattribute__` para no caer en una recursión infinita. Ver también la recursión que provoca hacerlo mal.

In [8]:
class Configuracion:
    POR_DEFECTO = {"idioma": "es", "zona": "Atlantic/Canary", "tema": "claro"}

    def __init__(self, **ajustes):
        self.__dict__.update(ajustes)

    def __getattr__(self, nombre):
        try:
            return self.POR_DEFECTO[nombre]
        except KeyError:
            raise AttributeError(f"ajuste desconocido: {nombre!r}") from None


c = Configuracion(tema="oscuro")
print(c.tema, c.idioma, c.zona, hasattr(c, "fuente"))
assert (c.tema, c.idioma) == ("oscuro", "es") and not hasattr(c, "fuente")


class Contador:
    def __init__(self):
        object.__setattr__(self, "accesos", {})
        self.valor = 10

    def __getattribute__(self, nombre):
        accesos = object.__getattribute__(self, "accesos")
        accesos[nombre] = accesos.get(nombre, 0) + 1
        return object.__getattribute__(self, nombre)

    def doble(self):
        return self.valor * 2


k = Contador()
k.valor + k.doble() + k.valor
print("accesos:", object.__getattribute__(k, "accesos"))
assert object.__getattribute__(k, "accesos") == {"valor": 3, "doble": 1}


class Recursiva:
    def __getattribute__(self, nombre):
        return self.__dict__[nombre]  # self.__dict__ vuelve a llamar a __getattribute__


try:
    Recursiva().x
except RecursionError as e:
    print("RecursionError:", e)

oscuro es Atlantic/Canary False
accesos: {'valor': 3, 'doble': 1}
RecursionError: maximum recursion depth exceeded


## 8. `__init_subclass__`: un registro de *plugins*

Crear una clase base `Exportador` que, con `__init_subclass__`, registre cada subclase bajo el formato que se pasa como argumento de clase (`class Csv(Exportador, formato="csv")`), rechace formatos repetidos y obligue a implementar `exportar`. Exportar unas mediciones con todos los formatos registrados sin escribir ningún `if`.

In [9]:
import json


class Exportador:
    registro = {}

    def __init_subclass__(cls, /, formato, **kwargs):
        super().__init_subclass__(**kwargs)
        if formato in Exportador.registro:
            raise TypeError(f"el formato {formato!r} ya lo exporta {Exportador.registro[formato].__name__}")
        if "exportar" not in cls.__dict__:
            raise TypeError(f"{cls.__name__} no implementa exportar()")
        cls.formato = formato
        Exportador.registro[formato] = cls


class Csv(Exportador, formato="csv"):
    def exportar(self, filas):
        return "\n".join([",".join(filas[0])] + [",".join(map(str, f.values())) for f in filas])


class Json(Exportador, formato="json"):
    def exportar(self, filas):
        return json.dumps(filas, ensure_ascii=False)


class Markdown(Exportador, formato="md"):
    def exportar(self, filas):
        cabecera = "| " + " | ".join(filas[0]) + " |"
        return "\n".join([cabecera, "|---" * len(filas[0]) + "|"] + ["| " + " | ".join(map(str, f.values())) + " |" for f in filas])


for intento in ("repetido", "incompleto"):
    try:
        if intento == "repetido":
            class OtroCsv(Exportador, formato="csv"):
                def exportar(self, filas): ...
        else:
            class Xml(Exportador, formato="xml"):
                pass
    except TypeError as e:
        print("TypeError:", e)

mediciones = [{"isla": "Lanzarote", "lluvia_mm": 110}, {"isla": "La Palma", "lluvia_mm": 740}]
for formato, clase in Exportador.registro.items():
    print(f"--- {formato}\n{clase().exportar(mediciones)}")
assert list(Exportador.registro) == ["csv", "json", "md"]

TypeError: el formato 'csv' ya lo exporta Csv
TypeError: Xml no implementa exportar()
--- csv
isla,lluvia_mm
Lanzarote,110
La Palma,740
--- json
[{"isla": "Lanzarote", "lluvia_mm": 110}, {"isla": "La Palma", "lluvia_mm": 740}]
--- md
| isla | lluvia_mm |
|---|---|
| Lanzarote | 110 |
| La Palma | 740 |


## 9. Las clases las crea `type`

Una clase es una instancia de su metaclase, que por defecto es `type`. Crear la misma clase con la sentencia `class` y llamando a `type(nombre, bases, espacio)`, comprobar que `type(type) is type`, y escribir una metaclase `Registrada` cuyo `__new__` registre todas las clases que crea, incluidas las de sus subclases, con el módulo y el orden de creación.

In [10]:
class IslaA:
    poblacion = 0

    def describir(self):
        return f"{type(self).__name__} con {self.poblacion} habitantes"


IslaB = type("IslaB", (), {"poblacion": 0, "describir": IslaA.describir})
for C in (IslaA, IslaB):
    print(f"{C.__name__}: type={type(C).__name__}, bases={[b.__name__ for b in C.__bases__]}, {C().describir()}")
print("type(type) is type:", type(type) is type, "| isinstance(int, type):", isinstance(int, type))
assert type(IslaB) is type and IslaB().describir() == "IslaB con 0 habitantes"


class Registrada(type):
    clases = []

    def __new__(mcs, nombre, bases, espacio, **kwargs):
        cls = super().__new__(mcs, nombre, bases, espacio, **kwargs)
        mcs.clases.append(cls)
        return cls


class Modelo(metaclass=Registrada):
    pass


class Usuario(Modelo):
    pass


class Reserva(Modelo):
    pass


print("creadas:", [c.__name__ for c in Registrada.clases], "| type(Usuario):", type(Usuario).__name__)
assert Registrada.clases == [Modelo, Usuario, Reserva]

IslaA: type=type, bases=['object'], IslaA con 0 habitantes
IslaB: type=type, bases=['object'], IslaB con 0 habitantes
type(type) is type: True | isinstance(int, type): True
creadas: ['Modelo', 'Usuario', 'Reserva'] | type(Usuario): Registrada


## 10. Metaclase de atributos obligatorios y `__prepare__`

Escribir una metaclase `Contrato` que, **al definir** la clase (no al instanciarla), compruebe que una subclase concreta define los atributos de la lista `requeridos` de su base y que no tienen el valor `None`. Usar además `__prepare__` para que el espacio de nombres sea un diccionario que prohíba definir dos veces el mismo método, un error que Python acepta en silencio.

In [11]:
class SinRepetidos(dict):
    def __setitem__(self, clave, valor):
        if clave in self and not clave.startswith("__"):
            raise TypeError(f"{clave!r} se define dos veces")
        super().__setitem__(clave, valor)


class Contrato(type):
    @classmethod
    def __prepare__(mcs, nombre, bases, **kwargs):
        return SinRepetidos()

    def __new__(mcs, nombre, bases, espacio, **kwargs):
        cls = super().__new__(mcs, nombre, bases, dict(espacio), **kwargs)
        faltan = [a for a in getattr(cls, "requeridos", ()) if getattr(cls, a, None) is None]
        if bases and faltan:
            raise TypeError(f"{nombre} no define {', '.join(faltan)}")
        return cls


class Sensor(metaclass=Contrato):
    requeridos = ("unidad", "minimo", "maximo")
    unidad = minimo = maximo = None


class Anemometro(Sensor):
    unidad, minimo, maximo = "km/h", 0, 250


definiciones = {
    "Pluviometro sin máximo": "class Pluviometro(Sensor):\n    unidad, minimo = 'mm', 0",
    "método repetido": (
        "class Barometro(Sensor):\n    unidad, minimo, maximo = 'hPa', 870, 1085\n"
        "    def leer(self): return 1013\n    def leer(self): return 0"
    ),
}
errores = []
for titulo, codigo in definiciones.items():
    try:
        exec(codigo, {"Sensor": Sensor})
    except TypeError as e:
        errores.append(str(e))
        print(f"{titulo:<23} TypeError: {e}")
print("Anemometro:", Anemometro.unidad, Anemometro.minimo, Anemometro.maximo)
assert errores == ["Pluviometro no define maximo", "'leer' se define dos veces"]

Pluviometro sin máximo  TypeError: Pluviometro no define maximo
método repetido         TypeError: 'leer' se define dos veces
Anemometro: km/h 0 250


## 11. Cuándo no usar una metaclase

Las metaclases se heredan y no se combinan: comprobar el `TypeError` de conflicto al heredar de dos clases con metaclases distintas (por ejemplo, `Modelo` del ejercicio 9 y una clase abstracta con `ABCMeta`). Resolver el registro del ejercicio 9 sin metaclase, con `__init_subclass__` y con un decorador de clase, y comprobar que dan el mismo resultado. El *singleton* con metaclase de *Patrones y SOLID* (ejercicio 5) es otro caso en el que bastaría un módulo o una función con caché.

In [12]:
from abc import ABC
from functools import cache

try:
    class ModeloAbstracto(Modelo, ABC):
        pass
except TypeError as e:
    print("TypeError:", e)


class BaseRegistro:
    clases = []

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        BaseRegistro.clases.append(cls)


class UsuarioB(BaseRegistro, ABC):
    pass


class ReservaB(BaseRegistro):
    pass


registradas = []


def registrar(cls):
    registradas.append(cls)
    return cls


@registrar
class UsuarioC:
    pass


@registrar
class ReservaC:
    pass


@cache
def configuracion():
    return {"zona": "Atlantic/Canary"}


nombres = [[c.__name__.rstrip("BC") for c in lista] for lista in (BaseRegistro.clases, registradas)]
print("__init_subclass__:", nombres[0], "| decorador:", nombres[1])
print("singleton con @cache:", configuracion() is configuracion())
assert nombres[0] == nombres[1] == ["Usuario", "Reserva"]

TypeError: metaclass conflict: the metaclass of a derived class must be a (non-strict) subclass of the metaclasses of all its bases
__init_subclass__: ['Usuario', 'Reserva'] | decorador: ['Usuario', 'Reserva']
singleton con @cache: True


## 12. `__class_getitem__`

`list[int]` no crea una lista, sino un `types.GenericAlias` que solo sirve para las anotaciones. Hacer que una clase `Caja` admita `Caja[int]` del mismo modo, y después escribir `Magnitud`, que con `__class_getitem__` devuelve (y guarda en caché) una subclase especializada en una unidad, de modo que `Magnitud["km"]` y `Magnitud["m"]` sean tipos distintos que no se pueden sumar entre sí.

In [13]:
from types import GenericAlias


class Caja:
    __class_getitem__ = classmethod(GenericAlias)

    def __init__(self, contenido):
        self.contenido = contenido


alias = Caja[int]
print(repr(alias), "| origen:", alias.__origin__.__name__, "| args:", alias.__args__, "| list[int]:", type(list[int]).__name__)
assert alias.__origin__ is Caja and alias(5).contenido == 5


class Magnitud:
    unidad = None
    _especializadas = {}

    def __class_getitem__(cls, unidad):
        if unidad not in cls._especializadas:
            cls._especializadas[unidad] = type(f"Magnitud_{unidad}", (cls,), {"unidad": unidad})
        return cls._especializadas[unidad]

    def __init__(self, valor):
        if self.unidad is None:
            raise TypeError("usa Magnitud[unidad](valor)")
        self.valor = valor

    def __add__(self, otra):
        if type(otra) is not type(self):
            return NotImplemented
        return type(self)(self.valor + otra.valor)

    def __repr__(self):
        return f"{self.valor} {self.unidad}"


Km, M = Magnitud["km"], Magnitud["m"]
print(Km(42) + Km(0.195), "|", M(800) + M(200), "| misma clase en caché:", Magnitud["km"] is Km)
try:
    Km(1) + M(500)
except TypeError as e:
    print("TypeError:", e)
assert Magnitud["km"] is Km and Km is not M and isinstance(Km(1), Magnitud)

__main__.Caja[int] | origen: Caja | args: (<class 'int'>,) | list[int]: GenericAlias
42.195 km | 1000 m | misma clase en caché: True
TypeError: unsupported operand type(s) for +: 'Magnitud_km' and 'Magnitud_m'


## 13. Gestor de contexto que suprime solo algunas excepciones

Escribir la clase `Tolerante(*tipos)`, cuyo `__exit__` devuelve `True` (suprime) solo si la excepción es de uno de los tipos indicados y además la guarda en una lista. Usarla para procesar lecturas de sensores donde algunas son ilegibles: los `ValueError` se toleran y el bucle continúa, pero un `ZeroDivisionError` sale del `with`. Comprobar el resultado contra `contextlib.suppress`.

In [14]:
from contextlib import suppress


class Tolerante:
    def __init__(self, *tipos):
        self.tipos = tipos
        self.ignoradas = []

    def __enter__(self):
        return self

    def __exit__(self, tipo, valor, traza):
        if tipo is not None and issubclass(tipo, self.tipos):
            self.ignoradas.append(f"{tipo.__name__}: {valor}")
            return True
        return False


lecturas = ["21.5", "22,1", "23.0", "", "error", "24.2"]
validas = []
contexto = Tolerante(ValueError)
for texto in lecturas:
    with contexto:
        validas.append(float(texto))
print("válidas:", validas)
print("ignoradas:", contexto.ignoradas)

validas_std = []
for texto in lecturas:
    with suppress(ValueError):
        validas_std.append(float(texto))
assert validas == validas_std == [21.5, 23.0, 24.2] and len(contexto.ignoradas) == 3

try:
    with Tolerante(ValueError):
        media = sum(validas) / 0
except ZeroDivisionError as e:
    print("No suprimida -> ZeroDivisionError:", e)

válidas: [21.5, 23.0, 24.2]
ignoradas: ["ValueError: could not convert string to float: '22,1'", "ValueError: could not convert string to float: ''", "ValueError: could not convert string to float: 'error'"]
No suprimida -> ZeroDivisionError: float division by zero
